In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

In [4]:
# retail_sales_data = pd.read_csv("Fashion_Retail_Sales.csv")
# retail_sales_data.head()

In [5]:
inventory_data = pd.read_csv("retail_store_inventory_data.csv")
inventory_data.head()

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer


In [6]:
inventory_data.shape

(73100, 15)

In [7]:
inventory_data.columns = inventory_data.columns.str.lower().str.replace(" ", "_")
inventory_data.head()

,date,store_id,product_id,category,region,inventory_level,units_sold,units_ordered,demand_forecast,price,discount,weather_condition,holiday/promotion,competitor_pricing,seasonality
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer


In [8]:
type(inventory_data)

pandas.core.frame.DataFrame

In [9]:
inventory_data['date'] = pd.to_datetime(inventory_data['date'])

In [10]:
inventory_data = inventory_data.sort_values(['product_id', 'date'])

In [11]:
inventory_data.columns

Index(['date', 'store_id', 'product_id', 'category', 'region',
       'inventory_level', 'units_sold', 'units_ordered', 'demand_forecast',
       'price', 'discount', 'weather_condition', 'holiday/promotion',
       'competitor_pricing', 'seasonality'],
      dtype='object')

In [12]:
inventory_data = inventory_data.rename(columns={'units_sold': 'sales'})

In [13]:
# On average, how many units of this product are sold per day over the last 7 days
inventory_data['sales_velocity_7d'] = (
    inventory_data.groupby('product_id')['sales']
    .rolling(7)
    .mean()
    .reset_index(level=0, drop=True)
)

In [14]:
inventory_data.set_index(['date'], inplace=True)

In [15]:
inventory_data['inventory_coverage_days'] = inventory_data['inventory_level'] / (inventory_data['sales_velocity_7d'] + 1e-5)

In [16]:
inventory_data['sales_velocity_14d'] = (
    inventory_data.groupby('product_id')['sales']
    .rolling(14)
    .mean()
    .reset_index(level=0, drop=True)
)

inventory_data['trend_score'] = inventory_data['sales_velocity_7d'] - inventory_data['sales_velocity_14d']

In [17]:
def recommendation(row):
    if row['trend_score'] > 0 and row['inventory_coverage_days'] < 5:
        return "Promote & Restock"
    elif row['inventory_coverage_days'] > 20:
        return "Deprioritize"
    else:
        return "Monitor"

inventory_data['recommendation'] = inventory_data.apply(recommendation, axis=1)

In [18]:
inventory_data['recommendation']

date
2022-01-01              Monitor
2022-01-01              Monitor
2022-01-01              Monitor
2022-01-01              Monitor
2022-01-01              Monitor
                    ...        
2024-01-01              Monitor
2024-01-01              Monitor
2024-01-01    Promote & Restock
2024-01-01    Promote & Restock
2024-01-01    Promote & Restock
Name: recommendation, Length: 73100, dtype: object

# gen ai layer

In [19]:
sample = inventory_data.iloc[-1]  # or filter for a specific product

context = f"""
Product: {sample['product_id']}
Sales velocity (7d): {sample['sales_velocity_7d']:.2f}
Trend score: {sample['trend_score']:.2f}
Inventory coverage: {sample['inventory_coverage_days']:.2f} days
Recommendation: {sample['recommendation']}
"""

In [20]:
prompt = f"""
You are a retail merchandising assistant.

Given the following data:
{context}

Explain:
1. Why this recommendation makes sense
2. What action should be taken
Keep it concise and business-focused.
"""

In [21]:
# from pydantic_core import validate_core_schema
from openai import OpenAI
client = OpenAI()

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}]
)

print(response.choices[0].message.content)

### 1. Why this recommendation makes sense:
The sales velocity of 140.29 indicates a high demand for product P0020 over the last week, suggesting it is a top-selling item. The trend score of 20.29 indicates that sales are increasing, reinforcing the need to capitalize on this momentum. With an inventory coverage of only 0.83 days, stock levels are critically low, which poses a risk of stockouts and lost sales opportunities. Therefore, promoting and restocking the product is essential to maintain sales performance and customer satisfaction.

### 2. What action should be taken:
Initiate marketing efforts to promote product P0020 immediately, leveraging its current popularity to boost visibility and sales. Simultaneously, place an urgent order for additional inventory to ensure sufficient stock is available to meet ongoing demand. Regularly monitor sales and inventory levels to adjust strategies as needed.


In [22]:
df_filtered = inventory_data[
    (inventory_data['recommendation'] != "Monitor")
].copy()

In [23]:
df_latest = df_filtered.sort_values('date').groupby('product_id').tail(1)

In [24]:
df_latest.head()

,store_id,product_id,category,region,inventory_level,sales,units_ordered,demand_forecast,price,discount,weather_condition,holiday/promotion,competitor_pricing,seasonality,sales_velocity_7d,inventory_coverage_days,sales_velocity_14d,trend_score,recommendation
date,,,,,,,,,,,,,,,,,,,
2023-12-29,S004,P0007,Toys,West,178,152,164,163.88,98.55,15,Snowy,1,98.40,Winter,184.142857,0.966641,165.142857,19.000000,Promote & Restock
2023-12-30,S003,P0009,Furniture,North,219,190,34,188.19,50.68,15,Cloudy,0,54.76,Spring,156.000000,1.403846,155.428571,0.571429,Promote & Restock
2023-12-31,S001,P0018,Clothing,East,303,0,30,2.19,81.52,10,Rainy,0,83.21,Spring,106.142857,2.854643,97.928571,8.214286,Promote & Restock
2023-12-31,S004,P0003,Furniture,West,169,33,153,51.71,11.59,15,Snowy,1,13.15,Spring,123.857143,1.364475,111.857143,12.000000,Promote & Restock
2023-12-31,S005,P0011,Furniture,South,68,13,82,8.68,69.32,15,Cloudy,1,72.28,Summer,155.142857,0.438306,145.928571,9.214286,Promote & Restock


In [25]:
results = []

for _, row in df_latest.iterrows():
    context = f"""
    Product: {row['product_id']}
    Sales velocity (7d): {row['sales_velocity_7d']:.2f}
    Trend score: {row['trend_score']:.2f}
    Inventory coverage: {row['inventory_coverage_days']:.2f} days
    Recommendation: {row['recommendation']}
    """

    prompt = f"""
    You are a retail merchandising assistant.

    {context}

    Explain briefly why and what action should be taken.
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )

    results.append({
        "product_id": row['product_id'],
        "recommendation": row['recommendation'],
        
        "llm_output": response.choices[0].message.content
    })

In [26]:
df_results = pd.DataFrame(results)
df_results.head()

,product_id,recommendation,llm_output
0,P0007,Promote & Restock,The product P0007 has a high sales velocity of...
1,P0009,Promote & Restock,The product P0009 has a sales velocity of 156 ...
2,P0018,Promote & Restock,The product P0018 has a high sales velocity of...
3,P0003,Promote & Restock,The product P0003 has a high sales velocity of...
4,P0011,Promote & Restock,The product P0011 is experiencing a high sales...


In [27]:
df_results.iloc[0]['llm_output']

'The product P0007 has a high sales velocity of 184.14 units sold in the past week, indicating strong customer demand. However, with only 0.97 days of inventory coverage, it is likely to run out of stock very soon. The trend score of 19.00 suggests a positive growth trend, reinforcing the idea that demand for this product is increasing.\n\n**Action:** To capitalize on this demand and avoid stockouts, it is recommended to promote the product to further boost sales and restock inventory immediately to maintain availability for customers. This will help maximize sales and prevent missed opportunities from potential buyers.'

In [28]:
df_results.to_csv("results.csv", index=False)

In [29]:
df_results

,product_id,recommendation,llm_output
0,P0007,Promote & Restock,The product P0007 has a high sales velocity of...
1,P0009,Promote & Restock,The product P0009 has a sales velocity of 156 ...
2,P0018,Promote & Restock,The product P0018 has a high sales velocity of...
3,P0003,Promote & Restock,The product P0003 has a high sales velocity of...
4,P0011,Promote & Restock,The product P0011 is experiencing a high sales...
5,P0013,Promote & Restock,The product P0013 shows a strong sales velocit...
6,P0012,Promote & Restock,The product P0012 is experiencing a strong sal...
7,P0010,Promote & Restock,The product P0010 is experiencing a high sales...
8,P0004,Promote & Restock,The product P0004 has a strong sales velocity ...
9,P0008,Promote & Restock,The product P0008 shows a high sales velocity ...


In [31]:
# langchain

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [32]:
from langchain.prompts import PromptTemplate

template = """
You are a retail merchandising assistant.

Product: {product_id}
Sales velocity (7d): {velocity}
Trend score: {trend}
Inventory coverage: {coverage} days
Recommendation: {recommendation}

Explain briefly:
1. Why this recommendation makes sense
2. What action should be taken
"""

prompt = PromptTemplate(
    input_variables=["product_id", "velocity", "trend", "coverage", "recommendation"],
    template=template
)

In [33]:
from langchain.chains import LLMChain

chain = LLMChain(llm=llm, prompt=prompt)

/var/folders/mg/nl4fflz92893yzhrmg7rwxth0000gn/T/ipykernel_3596/3249218726.py:3: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(llm=llm, prompt=prompt)


In [34]:
chain

LLMChain(verbose=False, prompt=PromptTemplate(input_variables=['coverage', 'product_id', 'recommendation', 'trend', 'velocity'], input_types={}, partial_variables={}, template='\nYou are a retail merchandising assistant.\n\nProduct: {product_id}\nSales velocity (7d): {velocity}\nTrend score: {trend}\nInventory coverage: {coverage} days\nRecommendation: {recommendation}\n\nExplain briefly:\n1. Why this recommendation makes sense\n2. What action should be taken\n'), llm=ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x12704ca90>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x1270628d0>, root_client=<openai.OpenAI object at 0x10e2188d0>, root_async_client=<openai.AsyncOpenAI object at 0x12704cc90>, model_name='gpt-4o-mini', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********')), output_parser=StrOutputParser(), llm_kwargs={})

In [35]:
output = chain.run({
    "product_id": row['product_id'],
    "velocity": round(row['sales_velocity_7d'], 2),
    "trend": round(row['trend_score'], 2),
    "coverage": round(row['inventory_coverage_days'], 2),
    "recommendation": row['recommendation']
})

print(output)

/var/folders/mg/nl4fflz92893yzhrmg7rwxth0000gn/T/ipykernel_3596/837430341.py:1: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  output = chain.run({


1. **Why this recommendation makes sense**: The sales velocity of 140.29 indicates that the product is selling quickly, suggesting strong demand. The trend score of 20.29 further reinforces this by showing that sales are increasing. However, with an inventory coverage of only 0.83 days, it means that the current stock will run out very soon, potentially leading to lost sales opportunities. Therefore, promoting the product can help capitalize on the current demand, while restocking ensures that inventory levels are sufficient to meet ongoing customer interest.

2. **What action should be taken**: The immediate actions should include launching a promotional campaign to increase visibility and drive sales, such as discounts, social media ads, or in-store displays. Simultaneously, a restock order should be placed to replenish inventory levels, ensuring that the product is available for customers as soon as possible. This dual approach will help maximize sales and maintain customer satisfac

In [ ]:
results = []

for _, row in df_latest.iterrows():
    output = chain.run({
        "product_id": row['product_id'],
        "velocity": round(row['sales_velocity_7d'], 2),
        "trend": round(row['trend_score'], 2),
        "coverage": round(row['inventory_coverage_days'], 2),
        "recommendation": row['recommendation']
    })

    results.append({
        "product_id": row['product_id'],
        "recommendation": row['recommendation'],
        "llm_output": output
    })

In [ ]:
results